## Testing Alpaca API

In [1]:
import os

from alpaca.data import (
    OptionHistoricalDataClient,
)
from alpaca.data.requests import OptionChainRequest
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("ALPACA_API_KEY")
api_secret = os.getenv("ALPACA_API_SECRET")

option_client = OptionHistoricalDataClient(api_key, api_secret)

In [4]:
from utils.alpaca_utils import flatten_option_data

req = OptionChainRequest(underlying_symbol="SPY",limit=1000,type="call")
chain_data = option_client.get_option_chain(request_params=req)

records = flatten_option_data(chain_data)
df = pl.DataFrame(records)


In [5]:
df.with_columns(date=pl.col("trade_timestamp").dt.date())

symbol,implied_volatility,greek_delta,greek_gamma,greek_rho,greek_theta,greek_vega,quote_timestamp,quote_bid_price,quote_bid_size,quote_bid_exchange,quote_ask_price,quote_ask_size,quote_ask_exchange,quote_conditions,quote_tape,trade_timestamp,trade_exchange,trade_price,trade_size,trade_id,trade_conditions,trade_tape,date
str,f64,f64,f64,f64,f64,f64,"datetime[μs, UTC]",f64,f64,str,f64,f64,str,str,null,"datetime[μs, UTC]",str,f64,f64,null,str,null,date
"""SPY260107C00697000""",0.101,0.0157,0.0063,0.0009,-0.0422,0.0244,2026-01-02 20:59:59.484545 UTC,0.03,2386.0,"""N""",0.04,2987.0,"""N""","""A""",null,2026-01-02 21:14:55.049417 UTC,"""M""",0.02,5.0,null,"""I""",null,2026-01-02
"""SPY260108C00682000""",0.1173,0.5709,0.0468,0.0423,-0.4507,0.2808,2026-01-02 20:59:59.737645 UTC,4.11,1.0,"""N""",4.12,1.0,"""N""","""A""",null,2026-01-02 21:14:48.867510 UTC,"""N""",4.32,15.0,null,"""I""",null,2026-01-02
"""SPY260108C00658000""",0.2177,0.9532,0.0063,0.0685,-0.2536,0.07,2026-01-02 20:59:59.002227 UTC,25.1,1.0,"""T""",26.38,1.0,"""T""","""A""",null,null,null,null,null,null,null,null,null
"""SPY260112C00570000""",null,null,null,null,null,null,2026-01-02 20:59:58.338959 UTC,111.79,14.0,"""C""",114.98,10.0,"""A""","""A""",null,null,null,null,null,null,null,null,null
"""SPY260107C00673000""",0.1488,0.8731,0.0226,0.0481,-0.3787,0.1288,2026-01-02 20:59:59.019011 UTC,10.8,53.0,"""C""",11.12,53.0,"""S""",""" """,null,2026-01-02 20:00:09.591765 UTC,"""N""",10.83,1.0,null,"""I""",null,2026-01-02
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SPY280616C00855000""",0.1425,0.2959,0.0023,4.3275,-0.0452,3.694,2026-01-02 20:59:59.402195 UTC,25.36,3.0,"""S""",25.53,76.0,"""W""",""" """,null,2025-12-31 20:05:57.995009 UTC,"""A""",25.48,1.0,null,"""I""",null,2025-12-31
"""SPY280616C00425000""",0.199,0.9736,0.0003,9.1092,-0.0405,0.6546,2026-01-02 20:59:59.691786 UTC,287.59,1.0,"""J""",298.83,1.0,"""S""",""" """,null,2025-12-23 15:22:16.853256 UTC,"""I""",297.35,29.0,null,"""f""",null,2025-12-23
"""SPY280616C00355000""",null,null,null,null,null,null,2026-01-02 20:59:58.154539 UTC,350.65,1.0,"""J""",353.27,20.0,"""I""",""" """,null,null,null,null,null,null,null,null,null


In [88]:
df.filter(pl.col("symbol").str.starts_with("SPY260109C000680"))

symbol,implied_volatility,greek_delta,greek_gamma,greek_rho,greek_theta,greek_vega,quote_timestamp,quote_bid_price,quote_bid_size,quote_bid_exchange,quote_ask_price,quote_ask_size,quote_ask_exchange,quote_conditions,quote_tape,trade_timestamp,trade_exchange,trade_price,trade_size,trade_id,trade_conditions,trade_tape
str,f64,f64,f64,f64,f64,f64,"datetime[μs, UTC]",f64,f64,str,f64,f64,str,str,null,"datetime[μs, UTC]",str,f64,f64,null,str,null


In [79]:
df.with_columns(strike=pl.col("symbol").str.slice(11, 5)).select("symbol", "strike").sort("strike",descending=True)

symbol,strike
str,str
"""ASTS260220C00150000""","""00150"""
"""ASTS270319C00150000""","""00150"""
"""ASTS270115C00150000""","""00150"""
"""ASTS271217C00150000""","""00150"""
"""ASTS260618C00150000""","""00150"""
…,…
"""ASTS260116C00002000""","""00002"""
"""ASTS260116C00002500""","""00002"""
"""ASTS260116C00001000""","""00001"""


## Testing post-no-preference's options data

https://www.dolthub.com/repositories/post-no-preference/options/data/master

In [9]:
import polars as pl

df = pl.read_parquet("../backtest/data/options_master.parquet")

In [113]:
df.filter(pl.col("act_symbol") == "GME",
pl.col("date")>"2023-12-19").group_by("act_symbol", "expiration","strike").agg(
    pl.count(),
    pl.col("date").min().alias("min_date"),
    pl.col("date").max().alias("max_date"),
).sort("count",descending=True)

C:\Users\Akshay Sadanandan\AppData\Local\Temp\ipykernel_40724\811418958.py:3: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count(),


act_symbol,expiration,strike,count,min_date,max_date
str,str,f64,u32,str,str
"""GME""","""2026-01-16""",20.0,56,"""2025-11-11""","""2026-01-02"""
"""GME""","""2025-04-17""",25.0,56,"""2025-02-10""","""2025-04-07"""
"""GME""","""2025-04-17""",23.0,56,"""2025-02-10""","""2025-04-07"""
"""GME""","""2026-01-16""",21.0,56,"""2025-11-11""","""2026-01-02"""
"""GME""","""2026-01-16""",19.0,56,"""2025-11-11""","""2026-01-02"""
…,…,…,…,…,…
"""GME""","""2024-06-14""",38.0,2,"""2024-05-15""","""2024-05-15"""
"""GME""","""2025-04-25""",29.5,2,"""2025-04-14""","""2025-04-14"""
"""GME""","""2024-08-23""",17.0,2,"""2024-08-12""","""2024-08-12"""


In [10]:
df.filter(pl.col("act_symbol") == "GME")['date'].min()

'2019-11-09'

In [40]:
df.filter(pl.col("act_symbol") == "AAPL").with_columns(
    opname=pl.col("act_symbol")
    + "_"
    + pl.col("expiration")
    + "_"
    + pl.col("strike").cast(pl.Int64).cast(pl.Utf8)
).group_by("opname").agg(pl.col("date").unique()).sort(pl.col("opname"))

opname,date
str,list[str]
"""AAPL_2019-02-22_145""","[""2019-02-09""]"
"""AAPL_2019-02-22_152""","[""2019-02-09""]"
"""AAPL_2019-02-22_157""","[""2019-02-09""]"
"""AAPL_2019-02-22_162""","[""2019-02-09""]"
"""AAPL_2019-02-22_167""","[""2019-02-09""]"
…,…
"""AAPL_2026-02-20_340""","[""2025-12-17"", ""2025-12-18"", … ""2026-01-02""]"
"""AAPL_2026-02-20_345""","[""2025-12-16""]"
"""AAPL_2026-02-20_350""","[""2025-12-17"", ""2025-12-22"", ""2026-01-02""]"


In [8]:
df.filter(pl.col("act_symbol") == "GME")

date,act_symbol,expiration,strike,call_put,bid,ask,vol,delta,gamma,theta,vega,rho
str,str,str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64
"""2019-11-09""","""GME""","""2019-11-22""",5.0,"""Call""",0.95,1.28,0.6926,0.9367,0.1505,-0.0039,0.0015,0.0018
"""2019-11-09""","""GME""","""2019-11-22""",5.0,"""Put""",0.0,0.12,0.7799,-0.085,0.1673,-0.0051,0.0019,-0.0002
"""2019-11-09""","""GME""","""2019-11-22""",5.5,"""Call""",0.53,0.86,0.6926,0.7951,0.344,-0.0086,0.0034,0.0016
"""2019-11-09""","""GME""","""2019-11-22""",5.5,"""Put""",0.0,0.32,0.7799,-0.2271,0.3255,-0.01,0.0036,-0.0005
"""2019-11-09""","""GME""","""2019-11-22""",6.0,"""Call""",0.24,0.53,0.6926,0.5725,0.4752,-0.0117,0.0047,0.0012
…,…,…,…,…,…,…,…,…,…,…,…,…
"""2026-01-02""","""GME""","""2026-02-20""",25.0,"""Put""",4.45,4.7,0.503,-0.8316,0.0728,-0.0081,0.0186,-0.0164
"""2026-01-02""","""GME""","""2026-02-20""",26.0,"""Call""",0.25,0.35,0.5426,0.1529,0.0575,-0.0102,0.0178,0.0038
"""2026-01-02""","""GME""","""2026-02-20""",26.0,"""Put""",5.3,6.2,0.6583,-0.7996,0.0598,-0.0125,0.0212,-0.0185
